In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

df = spark.read.format("parquet").load("abfss://bronze@dbproject.dfs.core.windows.net/customers")
df.display()

customer_id,first_name,last_name,email,city,state,_rescued_data
C00001,Emily,Mooney,rushjeff@ryan.org,Johnsonmouth,MS,null
C00002,Andrea,Sellers,mccoykiara@kelly.com,Stephenfort,WY,null
C00003,Craig,Hayes,rebeccamiller@yahoo.com,South Stephenshire,LA,null
C00004,Bryan,Scott,lawrence05@campbell.info,Chrisland,ND,null
C00005,Sean,Vasquez,carrie45@yahoo.com,East Dennistown,RI,null
C00006,Kevin,Mccarthy,traceyramos@gmail.com,North Matthew,IN,null
C00007,Amanda,Doyle,scottallen@gmail.com,Joneshaven,VA,null
C00008,Paul,Campos,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,null
C00009,Mary,Green,dennis03@yahoo.com,Kimberlyview,MD,null
C00010,James,Myers,charles58@murillo.net,West Hector,OK,null


In [0]:
df =df.drop(col("_rescued_data"))
df.display()

customer_id,first_name,last_name,email,city,state
C00001,Emily,Mooney,rushjeff@ryan.org,Johnsonmouth,MS
C00002,Andrea,Sellers,mccoykiara@kelly.com,Stephenfort,WY
C00003,Craig,Hayes,rebeccamiller@yahoo.com,South Stephenshire,LA
C00004,Bryan,Scott,lawrence05@campbell.info,Chrisland,ND
C00005,Sean,Vasquez,carrie45@yahoo.com,East Dennistown,RI
C00006,Kevin,Mccarthy,traceyramos@gmail.com,North Matthew,IN
C00007,Amanda,Doyle,scottallen@gmail.com,Joneshaven,VA
C00008,Paul,Campos,sullivanjeremy@horton-adams.com,South Nathanfurt,CT
C00009,Mary,Green,dennis03@yahoo.com,Kimberlyview,MD
C00010,James,Myers,charles58@murillo.net,West Hector,OK


In [0]:
df = df.withColumn("full_name", concat(col("first_name"), lit(" "), col("last_name")))
df = df.drop("first_name", "last_name")
df.display()

customer_id,email,city,state,full_name
C00001,rushjeff@ryan.org,Johnsonmouth,MS,Emily Mooney
C00002,mccoykiara@kelly.com,Stephenfort,WY,Andrea Sellers
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,Craig Hayes
C00004,lawrence05@campbell.info,Chrisland,ND,Bryan Scott
C00005,carrie45@yahoo.com,East Dennistown,RI,Sean Vasquez
C00006,traceyramos@gmail.com,North Matthew,IN,Kevin Mccarthy
C00007,scottallen@gmail.com,Joneshaven,VA,Amanda Doyle
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,Paul Campos
C00009,dennis03@yahoo.com,Kimberlyview,MD,Mary Green
C00010,charles58@murillo.net,West Hector,OK,James Myers


In [0]:
df = df.withColumn("domains", split(col("email"), "@")[1])
df.display()

customer_id,email,city,state,full_name,domains
C00001,rushjeff@ryan.org,Johnsonmouth,MS,Emily Mooney,ryan.org
C00002,mccoykiara@kelly.com,Stephenfort,WY,Andrea Sellers,kelly.com
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,Craig Hayes,yahoo.com
C00004,lawrence05@campbell.info,Chrisland,ND,Bryan Scott,campbell.info
C00005,carrie45@yahoo.com,East Dennistown,RI,Sean Vasquez,yahoo.com
C00006,traceyramos@gmail.com,North Matthew,IN,Kevin Mccarthy,gmail.com
C00007,scottallen@gmail.com,Joneshaven,VA,Amanda Doyle,gmail.com
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,Paul Campos,horton-adams.com
C00009,dennis03@yahoo.com,Kimberlyview,MD,Mary Green,yahoo.com
C00010,charles58@murillo.net,West Hector,OK,James Myers,murillo.net


In [0]:
df = df.withColumn(
    "email_category",
    when(col("domains") == "gmail.com", "Gmail")
    .when(col("domains") == "yahoo.com", "Yahoo")
    .when(col("domains") == "hotmail.com", "Hotmail")
    .when(col("domains") == "outlook.com", "Outlook")
    .otherwise("Other")
)
df.display()

customer_id,email,city,state,full_name,domains,email_category
C00001,rushjeff@ryan.org,Johnsonmouth,MS,Emily Mooney,ryan.org,Other
C00002,mccoykiara@kelly.com,Stephenfort,WY,Andrea Sellers,kelly.com,Other
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,Craig Hayes,yahoo.com,Yahoo
C00004,lawrence05@campbell.info,Chrisland,ND,Bryan Scott,campbell.info,Other
C00005,carrie45@yahoo.com,East Dennistown,RI,Sean Vasquez,yahoo.com,Yahoo
C00006,traceyramos@gmail.com,North Matthew,IN,Kevin Mccarthy,gmail.com,Gmail
C00007,scottallen@gmail.com,Joneshaven,VA,Amanda Doyle,gmail.com,Gmail
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,Paul Campos,horton-adams.com,Other
C00009,dennis03@yahoo.com,Kimberlyview,MD,Mary Green,yahoo.com,Yahoo
C00010,charles58@murillo.net,West Hector,OK,James Myers,murillo.net,Other


In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@dbproject.dfs.core.windows.net/customers")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS db_catalog.silver.customers_silver
USING DELTA
LOCATION 'abfss://silver@dbproject.dfs.core.windows.net/customers'